# LigandMPNN序列设计部分
## 首先获得待设计的骨架对应的地址，构建用于序列设计和打分的json文件

- 按照要求从PepSet中选取部分pdb，用于LigandMPNN大量设计

In [12]:
import os
import pandas as pd
from Bio.PDB import PDBParser
from collections import defaultdict

dataset = '/home/junjiechen/1_work/250401-Dpepalign/Benchmark/datasets/PepSet/Merged_PDBs'

# 记录dataset中每个复合物的肽段（L链）长度，输出每一种长度的pdb数目，保存在df中，注意：pdb只包含ATOM或者HETATM部分，没有SEQRES部分
length_dict = defaultdict(int)
parser = PDBParser(QUIET=True)

for pdb_file in os.listdir(dataset):
    if not pdb_file.endswith('.pdb'):
        continue
    structure = parser.get_structure(pdb_file, os.path.join(dataset, pdb_file))
    pep_len = None
    skip = False
    for model in structure:
        non_l_chains = [c for c in model if c.id != 'L']
        if len(non_l_chains) > 1:
            skip = True
            break
        for chain in model:
            if chain.id == 'L':
                pep_len = sum(1 for residue in chain if residue.id[0] == ' ')
                break
        if pep_len is not None:
            break
    if not skip and pep_len is not None:
        length_dict[pep_len] += 1

df = pd.DataFrame(
    sorted(length_dict.items()),
    columns=['Pep_Length', 'PDB_Count']
)
df


,Pep_Length,PDB_Count
0,5,16
1,6,15
2,7,14
3,8,17
4,9,12
5,10,13
6,11,14
7,12,13
8,13,14
9,14,9


In [16]:
# 接下来在每一种长度下随机选择3个pdb，复制到/home/junjiechen/1_work/250401-Dpepalign/Benchmark-Filter/datasets/PepSet_3per_length目录下，用于后续的测试
import random
import shutil
output_dir = '/home/junjiechen/1_work/250401-Dpepalign/Benchmark-Filter/datasets/PepSet_3per_length'
processed_database = '/home/junjiechen/1_work/250401-Dpepalign/Benchmark/datasets/PepSet/Processed_PDBs'
os.makedirs(output_dir, exist_ok=True)
length_to_pdbs = defaultdict(list)
for pdb_file in os.listdir(dataset):
    if not pdb_file.endswith('.pdb'):
        continue
    structure = parser.get_structure(pdb_file, os.path.join(dataset, pdb_file))
    pep_len = None
    skip = False
    for model in structure:
        non_l_chains = [c for c in model if c.id != 'L']
        if len(non_l_chains) > 1:
            skip = True
            break
        for chain in model:
            if chain.id == 'L':
                pep_len = sum(1 for residue in chain if residue.id[0] == ' ')
                break
        if pep_len is not None:
            break
    if not skip and pep_len is not None:
        length_to_pdbs[pep_len].append(pdb_file)
with open(os.path.join(output_dir, 'selected_pdbs.txt'), 'w') as f:
    for pep_len, pdbs in length_to_pdbs.items():
        f.write(f'Length {pep_len}:\n')
        selected_pdbs = random.sample(pdbs, min(3, len(pdbs)))
        for pdb_file in selected_pdbs:
            shutil.copy(os.path.join(processed_database, pdb_file), os.path.join(output_dir, pdb_file))
            f.write(f'{pdb_file}\n')
        f.write('\n')
    f.write('\n')


In [17]:
# 生成pdb路径文件
import os
import json

json_dict = {}
dir_path = "/home/junjiechen/1_work/250401-Dpepalign/Benchmark-Filter/datasets/PepSet_3per_length"

paths = []
for root, dirs, files in os.walk(f'{dir_path}'):
    for file in files:
        if file.endswith('.pdb'):
            full_path = os.path.join(root, file)
            paths.append(full_path)

# 编写json文件，格式为{"路径"：""},其中路径为complex.list中每一行的内容，去掉换行符
for path in paths:
        json_dict[path] = ""
with open(f"{dir_path}/Processed.json", "w") as json_file:
    json.dump(json_dict, json_file, indent=4)

## 接下来对序列设计结果进行筛选
- 由于对蛋白-多肽符合物，在采用LigandMPNN-HETATM版本进行设计时不能完全考虑overall_confidence，因此需要尝试多种筛选方法，并对筛选结果用结构预测模型进行预测，分析哪种筛选方式的设计成功率更高
- 这里的“设计成功率“为**ipTM>10的结果占比**

### 1. 总共设计10000条序列，并统计每一种序列出现的次数，按照次数降序，写入csv用于后续的Protenix结构预测验证

In [21]:
# 读取5d94.fa文件，统计不同序列出现的次数以及对应的overall_confidence的均值，最小值，最大值，保存到csv文件中
import pandas as pd
import os
from collections import defaultdict
from Bio.PDB import PDBParser
from Bio.SeqUtils import seq1

temperature = 0.2
res = '030'
pdb = '5d94'
method_path = f'./Output/{pdb}-{temperature}T/ligandmpnn_v_32_{res}_25'

pdb_path = "/home/junjiechen/1_work/250401-Dpepalign/Benchmark/datasets/PepSet/Merged_PDBs"
pdb_list = os.listdir(pdb_path)
print(pdb_list)
if f'{pdb}.pdb' in pdb_list:
    #读取pdb，将A链的单字母序列保存到变量sequence中，注意pdb只有ATOM或HETATM行
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure(pdb, f'{pdb_path}/{pdb}.pdb')
    for model in structure:
        for chain in model:
            if chain.id != 'L':
                pro_sequence = ''
                for residue in chain:
                    if residue.id[0] == ' ':
                        one_letter_resname = seq1(residue.get_resname())
                        pro_sequence += one_letter_resname


fa_file = f'{method_path}/seqs/{pdb}.fa'
sequences = []
confidences = defaultdict(list)

with open(fa_file, 'r') as f:
    for i, line in enumerate(f):
        if i >= 2:
            if line.startswith('>'):
                parts = line.strip().split(',')
                overall_confidence = float(parts[4].split('=')[1])
                ligand_confidence = float(parts[5].split('=')[1])
                seq_rec = float(parts[6].split('=')[1])
            else:
                seq = line.strip()
                sequences.append(seq)
                confidences[seq].append([overall_confidence, ligand_confidence, seq_rec])
sequence_stats = []
for seq, conf_list in confidences.items():
    count = len(conf_list)
    avg_overall_confidence = round(sum([conf[0] for conf in conf_list]) / count, 3)
    min_overall_confidence = min([conf[0] for conf in conf_list])
    max_overall_confidence = max([conf[0] for conf in conf_list])
    avg_ligand_confidence = round(sum([conf[1] for conf in conf_list]) / count, 3)
    min_ligand_confidence = min([conf[1] for conf in conf_list])
    max_ligand_confidence = max([conf[1] for conf in conf_list])
    avg_seq_rec = round(sum([conf[2] for conf in conf_list]) / count, 3)

    sequence_stats.append((pro_sequence, seq, count, avg_overall_confidence, min_overall_confidence, max_overall_confidence, avg_ligand_confidence, min_ligand_confidence, max_ligand_confidence, avg_seq_rec))



df = pd.DataFrame(
    sequence_stats,
    columns=['Pro_Sequence', 'Pep_Sequence', 'Count', 'Average_Overall_Confidence', 'Min_Overall_Confidence', 'Max_Overall_Confidence', 'Average_Ligand_Confidence', 'Min_Ligand_Confidence', 'Max_Ligand_Confidence', 'Average_Seq_Rec']
)
df = df.sort_values(by='Count', ascending=False)
df.index = range(1, len(df) + 1)
df.to_csv(f'{method_path}/5d94_ranked_by_number.csv', index=True)




['6fbk', '3pe4', '2nnu', '1nrl', '2xpn', '4xek', '5v3r', '3g2s', '6f6d', '5onp', '6co4', '6b27', '1j2x', '2ht9', '3v2o', '2ivz', '4leb', '5e33', '2khh', '5xxq', '1a0n', '4pge', '4pd1', '5di8', '2cch', '5dai', '4lk9', '6g5g', '5njc', '4fii', '2cny', '2e4h', '3wgx', '1czy', '4c5a', '5tgi', '6f0w', '1pmx', '6dei', '4j2c', '1f8h', '4dg3', '2vzg', '5fml', '4z2o', '1w80', '3et3', '2aij']


#### ps: 针对一种序列的overall_confidence打分情况，绘制直方图

In [ ]:
# 绘制指定序列的confidence分布直方图
import matplotlib.pyplot as plt
fa_file = './Output/5d94-0.2T/ligandmpnn_v_32_020_25/seqs/5d94.fa'
confidences = []
target_sequence = "RLEYEEVTEEEI"
with open(fa_file, 'r') as f:
    current_sequence = ''
    for i, line in enumerate(f):
        if i >= 2:
            if line.startswith('>'):
                parts = line.strip().split(',')
                overall_confidence = float(parts[4].split('=')[1])
            else:
                current_sequence = line.strip()
                if current_sequence == target_sequence:
                    confidences.append(overall_confidence)
plt.hist(confidences, bins=20, edgecolor='black')
plt.title('Overall Confidence Distribution')
plt.xlabel('Overall Confidence')
plt.ylabel('Number of Designs')
# plt.savefig('./Output/5d94-0.2T/ligandmpnn_v_32_020_25/overall_confidence_distribution.png')
# plt.close()

#### 随后进行Protenix结构预测

### 2. 总共设计10000条序列（现在不确定需要多少条），对得到的序列去重之后，重新这些序列进行打分，按照overall_confidence均值降序排序，选择top10用于后续的Protenix结构预测验证

In [5]:
# 从第三行开始读取fasta文件，将序列去重，相同的序列保留最上方的ID，将结果保存在unique文件中，将其中的蛋白名称增加下划线和id，保存unique文件
import json
import pandas as pd
from pathlib import Path

def dedup_fasta(src_path: str, dst_path: str) -> None:
    src = Path(src_path)
    dst = Path(dst_path)

    with src.open('r', encoding='utf-8') as f:
        lines = f.readlines()

    # Skip the first two lines (start reading from the 3rd line)
    lines = lines[2:]

    unique = []
    seen = set()
    header = None
    seq_lines = []

    def commit(hdr, seq_str):
        if not hdr:
            return
        if seq_str not in seen:
            seen.add(seq_str)
            unique.append((hdr, seq_str))

    for line in lines:
        line = line.rstrip('\n')
        if line.startswith('>'):
            # commit previous record
            commit(header, ''.join(seq_lines).strip())
            header = line
            seq_lines = []
        else:
            if line.strip():
                seq_lines.append(line.strip())
    # commit last record
    commit(header, ''.join(seq_lines).strip())

    with dst.open('w', encoding='utf-8') as out:
        for (hdr, seq) in unique:
            raw = hdr[1:].strip()
            parts = raw.split(',')
            parts = [p.strip() for p in parts]
            idx = parts[1].split('=')[1]
            name = parts[0] if parts else ''
            new_hdr = f">{name}_{idx}"
            out.write(new_hdr + '\n')
            for i in range(0, len(seq), 60):
                out.write(seq[i:i + 60] + '\n')

fasta_path = globals().get('fasta_path', '/home/junjiechen/1_work/250401-Dpepalign/Benchmark-Filter/Output/5d94-0.2T/ligandmpnn_v_32_030_25/seqs/5d94.fa')
tmp_path = globals().get('tmp_path', '/home/junjiechen/1_work/250401-Dpepalign/Benchmark-Filter/Output/5d94-0.2T/ligandmpnn_v_32_030_25/seqs/5d94_unique.fasta')
dedup_fasta(fasta_path, tmp_path)

#### 对去重后的序列进行打分
上面完成了序列去重，这一部分在backbone目录下找到去重后的序列，编写json文件作为打分的输入文件

In [6]:
# 编写json文件，键为tmp.fasta中的名称，值为““
import json

res = '030'
temperature = '0.2'

with open (tmp_path, 'r', encoding='utf-8') as f:
    lines = f.readlines()
path = f"/home/junjiechen/1_work/250401-Dpepalign/Benchmark-Filter/Output/5d94-{temperature}T/ligandmpnn_v_32_{res}_25/backbones"
seq_dict = {}
for line in lines:
    line = line.rstrip('\n')
    if line.startswith('>'):
        name = path + "/" + line[1:].strip() + ".pdb"
        seq_dict[name] = ""
with open(f'/home/junjiechen/1_work/250401-Dpepalign/Benchmark-Filter/Output/5d94-{temperature}T/ligandmpnn_v_32_{res}_25/score.json', 'w', encoding='utf-8') as f:
    json.dump(seq_dict, f, indent=4)

接下来进行序列打分，结果保存在score目录下

#### 分析打分结果，按照overall_confidence进行排序

In [7]:
# 按照 overall_confidence = exp[-mean_over_residues(log_probs for native sequence)] 计算 overall_confidence
import torch
import os
import pandas as pd

def overall_confidence_from_score_pt(pt_path: str):
    d = torch.load(pt_path, map_location="cpu")
    # inputs from score.py
    log_probs = torch.tensor(d["log_probs"], dtype=torch.float32)  # [N, L, 21]
    native_seq = torch.tensor(d["native_sequence"], dtype=torch.long)  # [L]
    mask = torch.tensor(d["mask"], dtype=torch.float32)  # [L]
    chain_mask = torch.tensor(d["chain_mask"], dtype=torch.float32)  # [L]
    m = mask * chain_mask  # [L]

    # broadcast native sequence across all samples N
    N, L, _ = log_probs.shape
    S_one_hot = torch.nn.functional.one_hot(native_seq, num_classes=21).float()  # [L, 21]
    S_one_hot = S_one_hot.unsqueeze(0).repeat(N, 1, 1)  # [N, L, 21]

    loss_per_pos = -(S_one_hot * log_probs).sum(-1)  # [N, L]
    loss = (loss_per_pos * m).sum(-1) / (m.sum() + 1e-8)  # [N]
    overall_confidence = torch.exp(-loss)  # [N]
    return overall_confidence[0].item()

def max_confidence_from_score_pt(pt_path: str):
    d = torch.load(pt_path, map_location="cpu")
    # inputs from score.py
    log_probs = torch.tensor(d["log_probs"], dtype=torch.float32)  # [N, L, 21]
    native_seq = torch.tensor(d["native_sequence"], dtype=torch.long)  # [L]
    mask = torch.tensor(d["mask"], dtype=torch.float32)  # [L]
    chain_mask = torch.tensor(d["chain_mask"], dtype=torch.float32)  # [L]
    m = mask * chain_mask  # [L]

    # broadcast native sequence across all samples N
    N, L, _ = log_probs.shape
    S_one_hot = torch.nn.functional.one_hot(native_seq, num_classes=21).float()  # [L, 21]
    S_one_hot = S_one_hot.unsqueeze(0).repeat(N, 1, 1)  # [N, L, 21]

    loss_per_pos = -(S_one_hot * log_probs).sum(-1)  # [N, L]
    loss = (loss_per_pos * m).sum(-1) / (m.sum() + 1e-8)  # [N]
    overall_confidence = torch.exp(-loss)  # [N]
    return overall_confidence.max().item()

dc = []
for file in os.listdir('/home/junjiechen/1_work/250401-Dpepalign/Benchmark-Filter/Output/5d94-0.2T/ligandmpnn_v_32_030_25/score/'):
    if file.endswith('.pt'):
        path = f'/home/junjiechen/1_work/250401-Dpepalign/Benchmark-Filter/Output/5d94-0.2T/ligandmpnn_v_32_030_25/score/{file}'
        conf = overall_confidence_from_score_pt(path)
        dc.append((file, conf))
df = pd.DataFrame(dc, columns=['pt', 'overall_confidence'])
df


,pt,overall_confidence
0,5d94_2845.pt,0.243210
1,5d94_7098.pt,0.245408
2,5d94_367.pt,0.275168
3,5d94_8892.pt,0.212331
4,5d94_162.pt,0.264364
...,...,...
2780,5d94_5675.pt,0.232430
2781,5d94_90.pt,0.275194
2782,5d94_353.pt,0.254723
2783,5d94_7429.pt,0.254018


In [8]:
df_sorted = df.sort_values(by='overall_confidence', ascending=False)
df_sorted

,pt,overall_confidence
392,5d94_59.pt,0.289116
1011,5d94_52.pt,0.287102
908,5d94_226.pt,0.285991
1152,5d94_31.pt,0.285131
340,5d94_88.pt,0.285104
...,...,...
2357,5d94_745.pt,0.198388
372,5d94_4457.pt,0.195777
1717,5d94_4898.pt,0.193839
934,5d94_100.pt,0.192625


最终输出的csv文件中第一列为MPNN生成的序列对应的文件名称，第二列为Protein序列，第三列为Pep序列，第四列为计算后的Overall confidence均值

In [9]:
# 将top10的文件名提取出来，在backbones目录下找到对应的pdb文件，将其中的L链提取出来，同时将蛋白链提取出来，保存到csv文件中，每一列分别为Pro_Sequence，Pep_Sequence和Average_Overall_Confidence
import pandas as pd
from Bio.PDB import PDBParser
from Bio.SeqUtils import seq1
import os
pdb = '5d94'
res = '030'
temperature = '0.2'


pdb_path = "/home/junjiechen/1_work/250401-Dpepalign/Benchmark/datasets/PepSet/Merged_PDBs"
pdb_list = os.listdir(pdb_path)
# print(pdb_list)
if f'{pdb}.pdb' in pdb_list:
    #读取pdb，将A链的单字母序列保存到变量sequence中，注意pdb只有ATOM或HETATM行
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure(pdb, f'{pdb_path}/{pdb}.pdb')
    for model in structure:
        for chain in model:
            if chain.id != 'L':
                pro_sequence = ''
                for residue in chain:
                    if residue.id[0] == ' ':
                        one_letter_resname = seq1(residue.get_resname())
                        pro_sequence += one_letter_resname


top10_df = df_sorted.head(10)
peptide_sequences = []
name = []
top10_df = top10_df.copy()
for complex in top10_df['pt']:
    complex_name = complex.replace('.pt', '')
    peptide_sequence = ''
    structure = parser.get_structure(complex_name, f'/home/junjiechen/1_work/250401-Dpepalign/Benchmark-Filter/Output/5d94-{temperature}T/ligandmpnn_v_32_{res}_25/backbones/{complex_name}.pdb')
    for model in structure:
        for chain in model:
            if chain.id == 'L':
                for residue in chain:
                    if residue.id[0] == ' ':
                        one_letter_resname = seq1(residue.get_resname())
                        peptide_sequence += one_letter_resname
    peptide_sequences.append(peptide_sequence)
    name.append(complex_name)

top10_df.loc[:, 'Designed_Sample'] = name
top10_df.loc[:, 'Pep_Sequence'] = peptide_sequences
top10_df.loc[:, 'Pro_Sequence'] = pro_sequence
top10_df = top10_df[['Designed_Sample', 'Pro_Sequence', 'Pep_Sequence', 'overall_confidence']]
top10_df = top10_df.rename(columns={'overall_confidence': 'Average_Overall_Confidence'}).reset_index(drop=True)
top10_df
top10_df.to_csv(f'/home/junjiechen/1_work/250401-Dpepalign/Benchmark-Filter/Output/5d94-{temperature}T/ligandmpnn_v_32_{res}_25/5d94_rescored_top10.csv', index=False)


### 3. 总共设计10000条序列，选择overall_confidence最大的10条不重复序列，这里没有采用重新打分的结果，而是采用10000条序列的统计结果

In [10]:
df3 = pd.read_csv("/home/junjiechen/1_work/250401-Dpepalign/Benchmark-Filter/Output/5d94-0.2T/ligandmpnn_v_32_030_25/5d94_ranked_by_number.csv")

# 按照max_overall_confidence排序，保存为新的csv文件
df3_sorted = df3.sort_values(by='Max_Overall_Confidence', ascending=False)
df3_sorted.to_csv("/home/junjiechen/1_work/250401-Dpepalign/Benchmark-Filter/Output/5d94-0.2T/ligandmpnn_v_32_030_25/5d94_ranked_by_max_overall_confidence.csv", index=False)